# آموزشِ صدا برای موتور محتوا

این نوت‌بوک **یک بار برای هر صدا** اجرا می‌شود.

## کاری که باید بکنید

۱. `Runtime` ← `Change runtime type` ← **`T4 GPU`**
۲. `Runtime` ← `Run all`

## اگر وسطش قطع شد

**فقط دوباره `Run all` بزنید.** از همان‌جا که بود ادامه می‌دهد.

همه‌چیز روی درایوِ شما ذخیره می‌شود، نه روی ماشینِ موقتیِ Colab:
وزن‌های دانلودشده، تکه‌های صوتی، و چک‌پوینتِ آموزش. پس قطعیِ اینترنت
چند دقیقه هزینه دارد، نه یک ساعت.

---

منطقِ قدم‌ها در مخزن است (`rvcpipe.py` و `dsprep.py`) و همین نوت‌بوک
آن‌ها را از آنجا می‌گیرد — پس یک نسخه بیشتر وجود ندارد.

## ۱ — کارتِ گرافیک و درایو

درایو **اول** وصل می‌شود، نه آخر: هرچه ساخته شود همان‌جا می‌نشیند تا
اگر اجرا قطع شد از دست نرود.

In [ ]:
import subprocess, sys, os, shutil, json, time

g = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True)
name = (g.stdout or b'').decode().strip()
if g.returncode != 0 or not name:
    raise SystemExit('GPU روشن نیست: Runtime ← Change runtime type ← T4 GPU')
print('کارت:', name)

from google.colab import drive
drive.mount('/content/drive')

## ۲ — تنظیمات

برای صدای رضوی همین‌طور که هست درست است. برای صدای دیگری `VOICE` و
`DRIVE_IDS` را عوض کنید.

In [ ]:
VOICE = 'razavi'

DRIVE_IDS = [
    '1YRI2p7Qv3hh2dcNPMZDmbNUel0XCYWKX',
    '1cBUasKKB2Q5JjLfpZfyjBC7ZNo72KAiD',
    '1izlhA9PRU0VWcmL-Gw7lFaW2LJ3nLUKv',
    '1QdJzUi8sk5LhuUqjHeCi9Kb4UgRYq4P5',
]

SR      = '40k'
EPOCHS  = 150
BATCH   = 8
# هر چند دوره چک‌پوینت ذخیره شود. کوچک‌تر = قطعی کمتر هزینه دارد.
SAVE_EVERY = 10

BASE    = '/content/drive/MyDrive/voice-models'
WORK    = BASE + '/work-' + VOICE

## ۳ — کد، و پوشهٔ کار روی درایو

`logs/` با پیوند به درایو وصل می‌شود. RVC خودش هر بار آخرین
چک‌پوینت را پیدا و از همان‌جا ادامه می‌دهد — پس اجرای دوباره یعنی
ادامه، نه شروع از صفر.

In [ ]:
os.chdir('/content')
if not os.path.isdir('/content/rvc'):
    subprocess.run(['git', 'clone', '--depth', '1', '-q',
                    'https://github.com/RVC-Project/'
                    'Retrieval-based-Voice-Conversion-WebUI',
                    '/content/rvc'], check=True)

RAW = 'https://raw.githubusercontent.com/mahdighandi1989/Content-Engine/main/tools'
for f in ('rvcpipe.py', 'dsprep.py'):
    subprocess.run(['curl', '-sSLf', RAW + '/' + f,
                    '-o', '/content/' + f], check=True)
sys.path.insert(0, '/content')
import rvcpipe as P, dsprep as D

ROOT = '/content/rvc'
for sub in ('logs', 'dataset', 'assets'):
    os.makedirs(WORK + '/' + sub, exist_ok=True)
# پیوند، نه کپی: مسیرهای داخلیِ RVC نسبی‌اند و باید زیرِ ریشه بمانند.
link = ROOT + '/logs'
if not os.path.islink(link):
    shutil.rmtree(link, ignore_errors=True)
    os.symlink(WORK + '/logs', link)
print('پوشهٔ کار:', WORK)
print('قدم‌ها:', [n for n, _ in P.steps(VOICE, '/x', ROOT, sr=SR)])

## ۴ — وابستگی‌ها

In [ ]:
# دو فهرست، هر دو از خودِ مخزن: یکی برای آموزشِ RVC، یکی برای فیلترِ
# موسیقی و تشخیصِ گوینده. اینجا نوشته نمی‌شوند تا با آنچه آزمایشگاه
# نصب می‌کند فرق نکنند.
deps = P.TRAIN_DEPS + D.DS_DEPS + ['gdown']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q']
               + deps, check=True)
print('نصب شد:', len(deps), 'بسته')

## ۵ — وزن‌های پایه

حدود ۵۵۰ مگابایت، و فقط بارِ اول. پروانه‌ها سنجیده شده‌اند: کدِ RVC
MIT · وزن‌ها ‏MIT · ContentVec ‏MIT · RMVPE ‏Apache-2.0.

In [ ]:
os.chdir(ROOT)
hf = shutil.which('hf') or shutil.which('huggingface-cli')
for cmd in P.assetCmds_(py=sys.executable, sr=SR, hf=hf or 'hf'):
    if cmd[:3] == [sys.executable, '-m', 'pip']:
        continue
    if cmd[0] in ('hf', 'huggingface-cli'):
        cmd[0] = hf or cmd[0]
    r = subprocess.run(cmd)
    if r.returncode:
        raise SystemExit('دانلود ناموفق: ' + ' '.join(cmd[:4]))
print('وزن‌ها آماده‌اند')

## ۶ — گرفتنِ ضبط‌ها

اگر قبلاً آمده باشند دوباره دانلود نمی‌شوند.

In [ ]:
import gdown
RAWDIR = WORK + '/raw'
os.makedirs(RAWDIR, exist_ok=True)
srcs = []
for i, fid in enumerate(DRIVE_IDS):
    dst = os.path.join(RAWDIR, 'in%d' % (i + 1))
    if not (os.path.exists(dst) and os.path.getsize(dst) > 100000):
        gdown.download(id=fid, output=dst, quiet=True)
    if os.path.exists(dst) and os.path.getsize(dst) > 100000:
        srcs.append(dst)
    else:
        print('نیامد (دسترسی؟):', fid)
print('%d از %d ضبط آماده است' % (len(srcs), len(DRIVE_IDS)))
if not srcs:
    raise SystemExit('هیچ فایلی نیامد — دسترسیِ اشتراکِ فایل‌ها را ببینید')

## ۷ — جداکردنِ موسیقی و ساختِ دیتاست

سه دروازه: فاصله‌های بلند (تیزر و میان‌برنامه)، کفِ هر تکه (موسیقیِ
زیرِ روایت)، و شباهت به گویندهٔ غالب (صدای کسِ دیگر).

اگر قبلاً ساخته شده باشد دوباره ساخته نمی‌شود.

In [ ]:
DS = WORK + '/dataset'
have = [f for f in os.listdir(DS) if f.endswith('.wav')]
if have:
    print('دیتاست از پیش آماده است: %d تکه' % len(have))
else:
    segs, rep = D.buildDataset_(srcs, DS, sampleDir=WORK)
    for row in rep['files']:
        print('%-22s %7.1f ثانیه → %3d تکه'
              % (row['file'][:22], row['seconds'], row.get('segments', 0)))
    print('\n' + rep['line'])
    if not segs:
        raise SystemExit('هیچ تکه‌ای نماند — ضبط‌ها را ببینید')

## ۸ — آموزش

طولانی‌ترین قدم. هر ۱۰ دوره چک‌پوینت روی درایو ذخیره می‌شود، پس اگر
قطع شد و دوباره `Run all` زدید، از همان‌جا ادامه می‌دهد.

In [ ]:
P.preLog_(ROOT, VOICE)
env = P.env(ROOT)
steps = P.steps(VOICE, DS, ROOT, sr=SR, f0method='rmvpe', epochs=EPOCHS,
                save_every=SAVE_EVERY, version='v2', gpus='0', n_p=2,
                batch=BATCH, py=sys.executable, latest=1)
done = os.path.join(WORK, 'logs', VOICE, '3_feature768')
for nm, cmd in steps:
    # قدم‌های آماده‌سازی اگر قبلاً انجام شده‌اند تکرار نمی‌شوند؛
    # آموزش همیشه صدا زده می‌شود چون خودش از چک‌پوینت ادامه می‌دهد.
    if nm in ('preprocess', 'extract_f0', 'extract_feature') \
            and os.path.isdir(done) and os.listdir(done):
        print('%s: از پیش انجام شده' % nm)
        continue
    if nm == 'train':
        info = P.preTrain_(ROOT, VOICE, sr=SR, version='v2')
        print('فهرستِ آموزش:', info)
        if not info['from_dataset']:
            raise SystemExit('فهرست خالی است — استخراج چیزی نساخت')
    t0 = time.time()
    print('\n=== %s ===' % nm, flush=True)
    r = subprocess.run(cmd, cwd=ROOT, env=env)
    print('%s: %ds' % (nm, time.time() - t0))
    if r.returncode:
        raise SystemExit('قدمِ «%s» شکست خورد (کد %d)' % (nm, r.returncode))

## ۹ — نتیجه

دو فایل: مدل و ایندکس. همین دو تا چیزی است که موتور به کار می‌برد.

In [ ]:
o = P.outputs(VOICE, ROOT)
if not os.path.exists(o['model']):
    raise SystemExit('آموزش تمام شد ولی مدلی ساخته نشد: ' + o['model'])
saved = [shutil.copy(o['model'], BASE)]
for f in sorted(os.listdir(o['index_dir'])):
    if f.endswith('.index'):
        saved.append(shutil.copy(os.path.join(o['index_dir'], f), BASE))
for s in saved:
    print('%8.1f مگابایت  %s' % (os.path.getsize(s) / 1048576, s))
print('\nتمام شد.')